# Lab 01 — Introductory Computer Vision (Tasks 1.1 – 9)
One code cell per task. Run the **Setup** cell first, then put your own file names in `PATHS`.
If an image is missing, the cell prints the required error message and continues with a
synthetic image, so every cell still runs.

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATHS = {
    "sukuna": "Sukuna.jpeg",        # Task 3
    "hires": "hires.jpg",           # Task 5
    "caption": "Sukuna.jpeg",       # Task 6
    "document": "document.jpg",     # Task 7 (grayscale document / high contrast)
    "img_a": "image1.jpg",          # Task 8
    "img_b": "image2.jpg",          # Task 8
    "stats": "Sukuna.jpeg",         # Task 9
}
os.makedirs("output", exist_ok=True)


def synthetic(kind="color", h=600, w=800):
    """Fallback images so every cell runs without the lab files."""
    rng = np.random.default_rng(sum(map(ord, kind)))
    if kind == "document":
        img = np.full((h, w), 235, np.uint8)
        for i, y in enumerate(range(60, h - 40, 45)):
            cv2.putText(img, f"Line {i + 1}: The quick brown fox jumps", (40, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, 30, 2)
        shade = np.tile(np.linspace(1.0, 0.45, w), (h, 1))      # uneven light
        return (img * shade).astype(np.uint8)
    img = np.zeros((h, w, 3), np.uint8)
    img[:] = rng.integers(40, 120, 3)
    for _ in range(12):
        c = tuple(int(v) for v in rng.integers(0, 255, 3))
        cv2.circle(img, (int(rng.integers(0, w)), int(rng.integers(0, h))),
                   int(rng.integers(30, 140)), c, -1)
    cv2.putText(img, kind.upper(), (30, h - 30), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 4)
    return img


def load(key, gray=False):
    path = PATHS[key]
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
    if img is None:
        print(f"Error: Image not found ({path}) -> using a synthetic image instead.")
        img = synthetic("document" if key == "document" else key)
        if gray and img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img


print("Setup done | OpenCV", cv2.__version__)

In [ ]:
# ===================== PART 1 — 1.1 Installation =====================
# Jupyter magic: installs into the kernel's own environment.
# (Skip in the exam if the libraries are already installed — no internet.)
%pip install -q opencv-python matplotlib

In [ ]:
# ===================== PART 1 — 1.2 Import & Validate =====================
# Import cv2 and matplotlib.pyplot, then prove both actually work.
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("OpenCV version    :", cv2.__version__)
print("Matplotlib version:", matplotlib.__version__)

test = np.zeros((100, 100, 3), np.uint8)
cv2.circle(test, (50, 50), 30, (0, 0, 255), -1)            # red in BGR
ok_cv = test[50, 50].tolist() == [0, 0, 255]
rgb = cv2.cvtColor(test, cv2.COLOR_BGR2RGB)
ok_conv = rgb[50, 50].tolist() == [255, 0, 0]
print("cv2 drawing works :", ok_cv)
print("BGR->RGB works    :", ok_conv)

plt.figure(figsize=(3, 3)); plt.imshow(rgb); plt.title("Validation: red circle"); plt.axis("off"); plt.show()
assert ok_cv and ok_conv, "Environment problem!"
print("Environment validated ✔")

In [ ]:
# ===================== TASK 1 — Object-Oriented Grocery Manager =====================
# GroceryManager with add_item(item, quantity, price), remove_item(item), view_list(),
# calculate_total() and basic error handling.
class GroceryManager:
    def __init__(self):
        self.items = {}                      # name -> {"quantity": int, "price": float}

    def add_item(self, item, quantity, price):
        if quantity <= 0 or price < 0:
            raise ValueError("Quantity must be > 0 and price must be >= 0")
        item = item.strip().lower()
        if item in self.items:               # same item again -> increase quantity
            self.items[item]["quantity"] += quantity
            self.items[item]["price"] = price
        else:
            self.items[item] = {"quantity": quantity, "price": price}
        print(f"Added {quantity} x {item} @ {price:.2f}")

    def remove_item(self, item):
        item = item.strip().lower()
        try:
            del self.items[item]
            print(f"Removed {item}")
        except KeyError:
            print(f"Error: '{item}' is not in the list, nothing removed.")

    def view_list(self):
        if not self.items:
            print("The grocery list is empty."); return
        print(f"{'Item':<12}{'Qty':>5}{'Price':>10}{'Subtotal':>12}")
        for name, d in self.items.items():
            print(f"{name:<12}{d['quantity']:>5}{d['price']:>10.2f}{d['quantity'] * d['price']:>12.2f}")

    def calculate_total(self):
        return sum(d["quantity"] * d["price"] for d in self.items.values())


gm = GroceryManager()
gm.add_item("Milk", 2, 250)
gm.add_item("Eggs", 12, 30)
gm.add_item("Bread", 1, 180)
gm.add_item("milk", 1, 250)                  # merges with existing milk
gm.remove_item("Butter")                     # error handling: does not exist
gm.remove_item("Bread")
gm.view_list()
print("Total cost:", gm.calculate_total())
try:
    gm.add_item("Rice", -1, 100)
except ValueError as e:
    print("Error:", e)

In [ ]:
# ===================== TASK 2 — Advanced Student Record System =====================
# Nested dict: ID -> {'Name', 'Major', 'Grades': [...]}.
# (1) function returning the student with the highest average grade
# (2) search function printing all students in a given major
students = {
    "K21-001": {"Name": "Ali",    "Major": "AI", "Grades": [88, 92, 79]},
    "K21-002": {"Name": "Sara",   "Major": "CS", "Grades": [95, 90, 93]},
    "K21-003": {"Name": "Hamza",  "Major": "AI", "Grades": [70, 85, 91]},
    "K21-004": {"Name": "Ayesha", "Major": "DS", "Grades": [99, 84, 90]},
    "K21-005": {"Name": "Usman",  "Major": "AI", "Grades": []},       # edge case
}


def top_student(records):
    best_name, best_avg = None, float("-inf")
    for sid, info in records.items():
        if not info["Grades"]:
            continue                                   # avoid division by zero
        avg = sum(info["Grades"]) / len(info["Grades"])
        if avg > best_avg:
            best_name, best_avg = info["Name"], avg
    return best_name, best_avg


def search_by_major(records, major):
    found = [(sid, i["Name"]) for sid, i in records.items() if i["Major"].lower() == major.lower()]
    if not found:
        print(f"No students enrolled in {major}"); return
    print(f"Students in {major}:")
    for sid, name in found:
        print(f"  {sid}: {name}")


name, avg = top_student(students)
print(f"Highest average: {name} ({avg:.2f})")
search_by_major(students, "AI")
search_by_major(students, "Physics")

In [ ]:
# ===================== TASK 3 — Safe Image Loading & RGB Visualization =====================
image_path = PATHS["sukuna"]
image = cv2.imread(image_path)

if image is None:
    print("Error: Image not found.")
    image = synthetic("sukuna")            # demo fallback so the rest of the cell runs

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # OpenCV = BGR, Matplotlib = RGB

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("King of Curses - Ryomen Sukuna", fontsize=16, color="darkred", pad=15)
plt.show()

In [ ]:
# ===================== TASK 4 — Interactive Geometry & Blank Canvas =====================
canvas = np.zeros((800, 800, 3), np.uint8)          # 800x800 black, 3 channels
h, w = canvas.shape[:2]
center = (w // 2, h // 2)                            # (x, y)

radii = [250, 200, 150, 100, 50]                    # draw big -> small so all stay visible
colors = [(0, 0, 255), (255, 255, 255)]             # red / white (BGR), alternating
for i, r in enumerate(radii):
    cv2.circle(canvas, center, r, colors[i % 2], -1)

R = radii[0]                                         # bounding box of outermost circle
top_left, bottom_right = (center[0] - R, center[1] - R), (center[0] + R, center[1] + R)
cv2.rectangle(canvas, top_left, bottom_right, (0, 255, 0), 3)

print("Canvas size        :", w, "x", h)
print("Exact centre (x, y):", center)
print("Bounding box       :", top_left, "->", bottom_right)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title("Target board", fontsize=14, color="navy"); plt.show()
cv2.imwrite("output/task4_target.png", canvas)

In [ ]:
# ===================== TASK 5 — Advanced Blurring & NumPy ROI Extraction =====================
img = load("hires")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
blurred = cv2.GaussianBlur(img_rgb, (25, 25), 0)    # heavy blur; kernel must be odd

h, w = img_rgb.shape[:2]
assert h >= 300 and w >= 300, "Image must be at least 300x300"
cy, cx = h // 2, w // 2
roi_orig = img_rgb[cy - 150:cy + 150, cx - 150:cx + 150]   # [rows=y, cols=x]
roi_blur = blurred[cy - 150:cy + 150, cx - 150:cx + 150]
print("ROI shapes:", roi_orig.shape, roi_blur.shape)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(roi_orig); plt.axis("off")
plt.title("Original ROI (300x300)", fontsize=14, color="darkgreen")
plt.subplot(1, 2, 2); plt.imshow(roi_blur); plt.axis("off")
plt.title("Gaussian Blur 25x25 ROI", fontsize=14, color="darkblue")
plt.tight_layout(); plt.show()

In [ ]:
# ===================== TASK 6 — Alpha Blending & Typography =====================
img = load("caption")
h, w = img.shape[:2]
overlay = img.copy()
cv2.rectangle(overlay, (0, int(h * 0.8)), (w, h), (255, 0, 0), -1)   # solid blue, bottom 20%

alpha = 0.4                                                          # overlay opacity
captioned = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

text = "Ryomen Sukuna"
font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, max(0.8, w / 600), 2
(tw, th), _ = cv2.getTextSize(text, font, scale, thick)
org = ((w - tw) // 2, int(h * 0.9) + th // 2)                        # centred in the band
cv2.putText(captioned, text, org, font, scale, (255, 255, 255), thick, cv2.LINE_AA)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Original"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(captioned, cv2.COLOR_BGR2RGB)); plt.title("Semi-transparent caption"); plt.axis("off")
plt.tight_layout(); plt.show()
cv2.imwrite("output/task6_caption.png", captioned)

In [ ]:
# ===================== TASK 7 — Matrix-Based Rotation & Adaptive Thresholding =====================
gray = load("document", gray=True)
_, global_th = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)          # one T for all pixels
adaptive_th = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 11, 2)            # local T per 11x11 block

h, w = adaptive_th.shape
M = cv2.getRotationMatrix2D((w / 2, h / 2), 45, 0.8)    # centre, +45 = CCW, scale 0.8
rotated = cv2.warpAffine(adaptive_th, M, (w, h), borderValue=0)   # dsize = (WIDTH, HEIGHT)
print("Rotation matrix:\n", np.round(M, 3))

plt.figure(figsize=(16, 5))
for i, (im, t) in enumerate([(gray, "Grayscale"), (global_th, "Global (T=127)"),
                             (adaptive_th, "Adaptive Gaussian"), (rotated, "Adaptive rotated 45°, scale 0.8")]):
    plt.subplot(1, 4, i + 1); plt.imshow(im, cmap="gray"); plt.title(t); plt.axis("off")
plt.tight_layout(); plt.show()
# Global fails where the page is darker (text and background both fall below T);
# adaptive uses a local threshold, so it handles the lighting gradient.
# Scale 0.8 is the requested value; the exact no-cut scale for 45° is
# min(w, h) / (w*|cos| + h*|sin|) — check visually that corners stay inside.

In [ ]:
# ===================== TASK 8 — Masking with Bitwise Logic =====================
a = cv2.resize(load("img_a"), (500, 500))
b = cv2.resize(load("img_b"), (500, 500))
if np.array_equal(a, b):                      # if both fell back to the same synthetic image
    b = cv2.flip(b, 1)

mask = np.zeros((500, 500), np.uint8)         # single-channel binary mask
star = np.array([[250, 60], [295, 190], [430, 190], [320, 270], [365, 410],
                 [250, 325], [135, 410], [180, 270], [70, 190], [205, 190]], np.int32)
cv2.fillPoly(mask, [star], 255)               # thick white polygon in the centre

fg = cv2.bitwise_and(a, a, mask=mask)                          # shape cut from image 1
bg = cv2.bitwise_and(b, b, mask=cv2.bitwise_not(mask))         # background from image 2
combined = cv2.bitwise_or(fg, bg)

titles = ["Image 1", "Image 2", "Mask", "Foreground (AND)", "Background (AND NOT)", "Combined (OR)"]
plt.figure(figsize=(15, 9))
for i, (im, t) in enumerate(zip([a, b, mask, fg, bg, combined], titles)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
    plt.title(t); plt.axis("off")
plt.tight_layout(); plt.show()
cv2.imwrite("output/task8_combined.png", combined)

In [ ]:
# ===================== TASK 9 — Statistical Image Profiling with Pandas =====================
img = load("stats")
B, G, R = cv2.split(img)                       # OpenCV order is B, G, R
df = pd.DataFrame({"Red": R.flatten(), "Green": G.flatten(), "Blue": B.flatten()})

summary = df.describe()                        # count, mean, std, min, quartiles, max
print(summary)
print("\nFocused summary:")
print(summary.loc[["mean", "min", "max", "std"]].round(2))
print("\nDominant channel (highest mean):", df.mean().idxmax())